## IPL win prediction model ##

In [1]:
# ================= IMPORTS =================
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings("ignore")

# ML
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

In [2]:
# ================= LOAD DATA =================
deliveries = pd.read_csv(r"C:\IPL_Match_Win_Predictor\data\deliveries.csv")
matches = pd.read_csv(r"C:\IPL_Match_Win_Predictor\data\matches.csv")

In [ ]:
# Merge datasets
df = deliveries.merge(matches, left_on='match_id', right_on='id')

# Only second innings (chasing team)
df = df[df['inning'] == 2]

# Current score
df['current_score'] = df.groupby('match_id')['total_runs'].cumsum()

# Runs left
df['runs_left'] = df['target_runs'] - df['current_score']

# Balls left
df['balls_left'] = 120 - (df['over'] * 6 + df['ball'])

# Wickets fallen
df['wickets'] = df.groupby('match_id')['player_dismissed'].transform(lambda x: x.notnull().cumsum())

# Wickets left
df['wickets_left'] = 10 - df['wickets']

# Required run rate
df['rrr'] = df['runs_left'] / (df['balls_left'] / 6)

# Final dataset
final_df = df[['batting_team',
               'bowling_team',
               'city',
               'runs_left',
               'balls_left',
               'wickets_left',
               'target_runs',
               'winner']]

In [ ]:
final_df.dropna(inplace=True)

# 1 if chasing team wins
final_df['result'] = (final_df['batting_team'] == final_df['winner']).astype(int)

final_df.drop(columns=['winner'], inplace=True)

X = final_df.drop(columns=['result'])
y = final_df['result']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
categorical_cols = ['batting_team', 'bowling_team', 'city']

trf = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

pipe = Pipeline([
    ('preprocessing', trf),
    ('model', LogisticRegression(max_iter=1000))
])

pipe.fit(X_train, y_train)

In [ ]:
y_pred = pipe.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

pipe.predict_proba(X_test.iloc[[0]])

In [ ]:
with open("ipl_win_predictor.pkl", "wb") as file:
    pickle.dump(pipe, file)

print("Model saved successfully!")